In [1]:
import signal
import wandb
import torch
import os 

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize

from codefiles.helpers import is_running_in_notebook  # for reloading modules instead of restarting kernel
if is_running_in_notebook():
    from codefiles import helpers
    from codefiles import architecture
    from codefiles import encoders
    from codefiles import transformer
    from codefiles.lightningdatamodules import mimic_symile
    from codefiles.lightningmodules import mimic
    import importlib
    importlib.reload(helpers)
    importlib.reload(architecture)
    importlib.reload(encoders)
    importlib.reload(transformer)
from codefiles.helpers import set_all_seeds, signal_handler, build_model
from codefiles.lightningmodules.mimic import MIMIC_Lightning_Module
from codefiles.lightningdatamodules.mimic_symile import MIMIC_Symile_Datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")

def main(cfg) -> None:
    seed = 420
    set_all_seeds(seed=seed)

    wandb.init(
        project="simple_mml_baseline",
        # config={key: value for key, value in cfg.items()},
    )

    model = build_model()

    lightningmodule = MIMIC_Lightning_Module(
        model=model
    )
    datamodule = MIMIC_Symile_Datamodule(
        seed=seed,
    )

    trainer = pl.Trainer(
        logger=WandbLogger(project="simple_mml_baseline", dir="wandb/"),
        log_every_n_steps=1,
        # accelerator='gpu',
        devices=1,
        max_epochs=50,
        # precision="bf16-mixed"
    )

    trainer.fit(lightningmodule, datamodule)
    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    signal.signal(signal.SIGINT, signal_handler)
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")

    # cfg = {key: value for key, value in cfg.items()}
    main(cfg)

/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
Seed set to 420


AttributeError: 'Encoder_Image' object has no attribute '_init_weights'